# 1) Content Extraction

In [ ]:
#content_extraction.py

from pptx import Presentation

def extract_presentation_content(ppt_path):

    prs = Presentation(ppt_path)

    slides = []

    for slide in prs.slides:

        slide_data = {
            "title": "",
            "content": []
        }

        for shape in slide.shapes:

            if shape.has_text_frame:

                text = shape.text.strip()

                if not text:
                    continue

                if shape == slide.shapes.title:
                    slide_data["title"] = text
                else:
                    slide_data["content"].append(text)

        slides.append(slide_data)

    return slides

# 2) Template Analysis

In [ ]:
#template_analysis.py

import os
from pptx import Presentation

def analyze_templates(template_folder):

    templates = []

    for file in os.listdir(template_folder):

        if file.endswith(".pptx"):

            path = os.path.join(template_folder, file)

            prs = Presentation(path)

            layouts = []

            for layout in prs.slide_layouts:

                layouts.append(layout.name)

            templates.append({
                "name": file,
                "path": path,
                "layouts": layouts
            })

    return templates

# 3)Build RAG Vector Store

In [ ]:
#template_analysis.py

from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document


def create_vector_store(slides, templates):

    docs = []

    for slide in slides:

        text = f"""
        Slide Title: {slide['title']}
        Content: {' '.join(slide['content'])}
        """

        docs.append(Document(page_content=text, metadata={"type": "slide"}))

    for template in templates:

        text = f"""
        Template Name: {template['name']}
        Layouts: {template['layouts']}
        """

        docs.append(Document(page_content=text, metadata={"type": "template"}))

    embeddings = OpenAIEmbeddings()

    vector_db = FAISS.from_documents(docs, embeddings)

    return vector_db

# 4) Template Selection with RAG

In [ ]:
#template_selection.py


from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA


def select_template(vector_db):

    retriever = vector_db.as_retriever()

    llm = ChatOpenAI(model="gpt-4o-mini")

    chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever
    )

    query = """
    Analyze the slide structure and select the best PPT template
    that supports the slide layout and content.

    Return only the template name.
    """

    result = chain.run(query)

    return result

# 5) Final Presentation Generation

In [ ]:
#presentation_generation.py

from pptx import Presentation

def generate_presentation(template_path, slides, output):

    prs = Presentation(template_path)

    for slide_data in slides:

        layout = prs.slide_layouts[1]

        slide = prs.slides.add_slide(layout)

        slide.shapes.title.text = slide_data["title"]

        content = "\n".join(slide_data["content"])

        try:
            slide.placeholders[1].text = content
        except:
            pass

    prs.save(output)

# 6) End-to-End Pipeline

In [ ]:
#main.py

from content_extraction import extract_presentation_content
from template_analysis import analyze_templates
from rag_store import create_vector_store
from template_selection import select_template
from presentation_generation import generate_presentation


INPUT_PPT = "input/input.pptx"
TEMPLATE_FOLDER = "templates"
OUTPUT_PPT = "output/final_presentation.pptx"


def run():

    print("Extracting slides")

    slides = extract_presentation_content(INPUT_PPT)

    print("Analyzing templates")

    templates = analyze_templates(TEMPLATE_FOLDER)

    print("Building RAG store")

    vector_db = create_vector_store(slides, templates)

    print("Selecting best template")

    template_name = select_template(vector_db)

    template_path = f"{TEMPLATE_FOLDER}/{template_name}"

    print("Generating presentation")

    generate_presentation(template_path, slides, OUTPUT_PPT)


if __name__ == "__main__":
    run()